# **ANÁLISIS EXPLORATORIO DE DATOS PARTE 1**

1. Funciones de resumen

2. Uso de función map

3. Uso de función groupby

4. Uso de multi-índices

5. Uso de la función sort

<img src="./imgs/foto-dia-03.jpg" width="500px" height="300px">


En la anterior clase aprendimos a seleccionar datos relevantes de un DataFrame o Serie. Extraer los datos correctos de nuestra representación de datos es fundamental para realizar el trabajo, como vimos en los ejercicios.

Sin embargo, los datos no siempre están en el formato que deseamos. A veces tenemos que hacer algo más de trabajo nosotros mismos para reformatearlos para la tarea que tenemos entre manos.


In [91]:
import pandas as pd

### **1. Funciones de resumen**


Pandas proporciona muchas "funciones de resumen" simples (no es un nombre oficial) que reestructuran los datos de alguna manera útil. Por ejemplo, el método describe():


In [92]:
compras = pd.read_csv('datasets/compras_centro_comercial.csv')
compras

,Customer ID,Age,Gender,Annual Income,Spending Score
0,d410ea53-6661-42a9-ad3a-f554b05fd2a7,30,Male,151479,89
1,1770b26f-493f-46b6-837f-4237fb5a314e,58,Female,185088,95
2,e81aa8eb-1767-4b77-87ce-1620dc732c5e,62,Female,70912,76
3,9795712a-ad19-47bf-8886-4f997d6046e3,23,Male,55460,57
4,64139426-2226-4cd6-bf09-91bce4b4db5e,24,Male,153752,76
...,...,...,...,...,...
15074,a0504768-a85f-4930-ac24-55bc8e4fec9e,29,Female,97723,30
15075,a08c4e0e-d1fe-48e7-9366-aab11ae409cd,22,Male,73361,74
15076,0e87c25a-268c-401a-8ba1-7111dcde6f1a,18,Female,112337,48
15077,5f388cbe-3373-4e16-b743-38f508f2249f,26,Female,94312,5


Este método genera un resumen de estadísticas de los atributos de la columna dada. Su salida cambia en función del tipo de datos de entrada. La salida anterior sólo tiene sentido para datos numéricos; para datos de string esto es lo que obtenemos:


In [93]:
compras['Annual Income'].describe()

count     15079.000000
mean     109742.880562
std       52249.425866
min       20022.000000
25%       64141.000000
50%      109190.000000
75%      155008.000000
max      199974.000000
Name: Annual Income, dtype: float64

In [94]:
compras['Gender'].describe()

count     15079
unique        2
top        Male
freq       7595
Name: Gender, dtype: object

Si deseas obtener alguna estadística de resumen simple sobre una columna en un DataFrame o una Serie, normalmente hay una función de pandas útil que lo hace posible.

Por ejemplo, para ver la media de los ingresos anuales de los compradores, podemos usar la función mean():


In [95]:
print(compras['Annual Income'].mean())
print(compras['Annual Income'].median())
print(compras['Annual Income'].std())
print(compras['Annual Income'].quantile(0.25))

109742.88056237152
109190.0
52249.425866344674
64141.0


Para ver una lista de valores únicos, podemos utilizar el método unique():


In [96]:
compras['Gender'].unique()

array(['Male', 'Female'], dtype=object)

Para ver una lista de valores únicos y la frecuencia con la que aparecen en el conjunto de datos, podemos utilizar el método value_counts():


In [97]:
compras['Age'].value_counts()

Age
86    249
65    235
51    234
55    229
69    228
     ... 
82    183
61    182
23    175
68    174
73    173
Name: count, Length: 73, dtype: int64

In [98]:
compras['Gender'].value_counts()

Gender
Male      7595
Female    7484
Name: count, dtype: int64

>#### **Detección de outliers con IQR**
> 
>Para cada columna numérica calculamos:
>
>- Q1 (percentil 25)
>- Q3 (percentil 75)
>- IQR = Q3 - Q1
>
>Consideramos outlier cualquier valor fuera del rango:
>
>- Límite inferior: $Q1 - 1.5 \times IQR$
>- Límite superior: $Q3 + 1.5 \times IQR$

In [100]:
# seleccionar solo las columnas numericas del df compras
columnas_numericas = compras.select_dtypes(include='number').columns

# lista donde se va a guardar el resumen de outliers por cada columna
resumen_outliers = []

# recorrer cada columna numerica para calcular limites de outliers
for col in columnas_numericas:
    # percentil 25%
    q1 = compras[col].quantile(0.25)

    # percentil 75%
    q3 = compras[col].quantile(0.75)

    # cálculo IQR
    iqr = q3 - q1

    # limites pora detectar outliers: li = q1 - 1.5(iqr) | ls = q3 + 1.5(iqr)
    limite_inferior = max(0, q1 - 1.5 * iqr)
    limite_superior = q3 + 1.5 * iqr

    # filtrar la filas con valores fuera de los límites (outliers)
    outliers = compras[(compras[col] < limite_inferior) | (compras[col] > limite_superior)]

    # guardar el resumen de estadísticas en la lista
    resumen_outliers.append({
        'columna': col,
        'Q1': q1,
        'Q3': q3,
        'IQR': iqr,
        'limite_inferior': limite_inferior,
        'limite_superior': limite_superior,
        'cantidad_outliers': outliers.shape[0]
    })

# convertir la lista de dicciioinario en df y ordenar por cantidad de outliers (mayor a menor)
pd.DataFrame(resumen_outliers).sort_values('cantidad_outliers', ascending=False)

,columna,Q1,Q3,IQR,limite_inferior,limite_superior,cantidad_outliers
0,Age,36.0,72.0,36.0,0,126.0,0
1,Annual Income,64141.0,155008.0,90867.0,0,291308.5,0
2,Spending Score,26.0,75.0,49.0,0,148.5,0


### **2. Uso de función map**


Un mapa es un término, tomado de las matemáticas, para una función que toma un conjunto de valores y los "mapea" a otro conjunto de valores. En la ciencia de datos, a menudo necesitamos crear nuevas representaciones a partir de datos existentes, o transformar datos del formato actual al formato que queremos que tengan más adelante. Los mapas son los que se encargan de este trabajo, por lo que son extremadamente importantes para realizarlo.

Hay dos métodos de mapeo que usarás a menudo.

map() es el primero, y ligeramente más simple.

Por ejemplo, imáginate que queremos normalizar el valor del campo `Annual Income` en función de la media, de este modo los salarios que sean negativos serán los que estén por debajo de la media y viceversa.


In [108]:
media_salario = compras['Annual Income'].mean()
salarios_media = compras['Annual Income'].map(lambda salario: salario - media_salario)
print(salarios_media[salarios_media > 0].shape)
print(round(salarios_media, 2))

(7486,)
0        41736.12
1        75345.12
2       -38830.88
3       -54282.88
4        44009.12
           ...   
15074   -12019.88
15075   -36381.88
15076     2594.12
15077   -15430.88
15078   -31697.88
Name: Annual Income, Length: 15079, dtype: float64


La función que se le pasa a map() debe esperar un único valor de la Serie (un valor de salario, en el ejemplo anterior), y devolver una versión transformada de ese valor. map() devuelve una nueva Serie en la que todos los valores han sido transformados por su función.

apply() es el método equivalente si queremos transformar todo un DataFrame llamando a un método personalizado en cada fila.


In [109]:
def normalizar_puntuacion_compra (row):
    row['Spending Score'] = row['Spending Score'] // 10
    return row

compras.apply(normalizar_puntuacion_compra, axis='columns')

,Customer ID,Age,Gender,Annual Income,Spending Score
0,d410ea53-6661-42a9-ad3a-f554b05fd2a7,30,Male,151479,8
1,1770b26f-493f-46b6-837f-4237fb5a314e,58,Female,185088,9
2,e81aa8eb-1767-4b77-87ce-1620dc732c5e,62,Female,70912,7
3,9795712a-ad19-47bf-8886-4f997d6046e3,23,Male,55460,5
4,64139426-2226-4cd6-bf09-91bce4b4db5e,24,Male,153752,7
...,...,...,...,...,...
15074,a0504768-a85f-4930-ac24-55bc8e4fec9e,29,Female,97723,3
15075,a08c4e0e-d1fe-48e7-9366-aab11ae409cd,22,Male,73361,7
15076,0e87c25a-268c-401a-8ba1-7111dcde6f1a,18,Female,112337,4
15077,5f388cbe-3373-4e16-b743-38f508f2249f,26,Female,94312,0


Si hubierámos llamado a compras.apply() con axis='index', entonces en lugar de pasar una función para transformar cada fila, tendríamos que dar una función para transformar cada columna.

Date cuenta que map() y apply() devuelven Series y DataFrames nuevos y transformados, respectivamente. No modifican los datos originales.


In [110]:
def suma_valores(column):
    if column.dtype == 'int64':
        column = column - 100
    if column.dtype == 'object':
        column = column.str.upper()
    return column

compras.apply(suma_valores, axis='index')

,Customer ID,Age,Gender,Annual Income,Spending Score
0,D410EA53-6661-42A9-AD3A-F554B05FD2A7,-70,MALE,151379,-11
1,1770B26F-493F-46B6-837F-4237FB5A314E,-42,FEMALE,184988,-5
2,E81AA8EB-1767-4B77-87CE-1620DC732C5E,-38,FEMALE,70812,-24
3,9795712A-AD19-47BF-8886-4F997D6046E3,-77,MALE,55360,-43
4,64139426-2226-4CD6-BF09-91BCE4B4DB5E,-76,MALE,153652,-24
...,...,...,...,...,...
15074,A0504768-A85F-4930-AC24-55BC8E4FEC9E,-71,FEMALE,97623,-70
15075,A08C4E0E-D1FE-48E7-9366-AAB11AE409CD,-78,MALE,73261,-26
15076,0E87C25A-268C-401A-8BA1-7111DCDE6F1A,-82,FEMALE,112237,-52
15077,5F388CBE-3373-4E16-B743-38F508F2249F,-74,FEMALE,94212,-95


### **3. Uso de función groupby**


Los mapas nos permiten transformar los datos en un DataFrame o Serie un valor a la vez para una columna entera. Sin embargo, a menudo queremos agrupar nuestros datos, y luego hacer algo específico para el grupo en el que se encuentran los datos.

Para esto tenemos la operación groupby().


In [113]:
compras.groupby('Gender')['Gender'].count()

Gender
Female    7484
Male      7595
Name: Gender, dtype: int64

### `size()` en pandas

El método `size()` cuenta la cantidad de elementos por grupo. Se utiliza mucho junto con `groupby()` para conocer cuántas filas hay en cada categoría.

Sintaxis general:

`df.groupby("columna").size()`


In [114]:
compras.groupby('Gender').size()

Gender
Female    7484
Male      7595
dtype: int64

> En este ejemplo, `size()` devuelve el total de registros por cada combinación de grupo. A diferencia de `count()`, este método no ignora valores nulos porque cuenta filas, no valores de una columna específica.


La función groupby() internamente crea un grupo de compras que comparten el mismo valor para la columna `gender`. A continuación, para cada uno de estos grupos, tomamos la columna gender() y contamos cuántas veces aparecía. value_counts() es sólo un atajo para esta operación groupby().

Podemos utilizar cualquiera de las funciones de resumen que hemos utilizado antes con estos datos. Por ejemplo, para obtener el salario mínimo que cobran las mujeres y hombres podemos hacer lo siguiente:


In [119]:
compras.groupby('Gender')['Annual Income'].max()

Gender
Female    199963
Male      199974
Name: Annual Income, dtype: int64

Se puede pensar en cada grupo que generamos como una porción de nuestro DataFrame que contiene sólo datos con valores que coinciden. Podemos acceder directamente a este DataFrame utilizando el método apply(), y podemos manipular los datos de la forma que creamos conveniente. Por ejemplo, si queremos obtener el primer salario del primer hombre y de la primera mujer podemos hacer lo siguiente.


In [121]:
compras.groupby('Gender').apply(lambda df:df['Annual Income'].iloc[0])

C:\Users\ANALISIS DE DATOS\AppData\Local\Temp\ipykernel_10056\2866601727.py:1: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  compras.groupby('Gender').apply(lambda df:df['Annual Income'].iloc[0])


Gender
Female    185088
Male      151479
dtype: int64

Para un control aún más preciso, también se puede agrupar por más de una columna. Por ejemplo, así es como seleccionaríamos los salarios mínimos y máximos agrupados por género y edad:


In [126]:
compras.groupby(['Age', 'Gender'])['Annual Income'].min().head(50)

Age  Gender
18   Female    22036
     Male      31004
19   Female    20555
     Male      22196
20   Female    22523
     Male      23370
21   Female    20897
     Male      22396
22   Female    24596
     Male      20199
23   Female    22487
     Male      20440
24   Female    23171
     Male      20026
25   Female    20022
     Male      21068
26   Female    20375
     Male      21797
27   Female    23763
     Male      20177
28   Female    20532
     Male      20859
29   Female    22540
     Male      21407
30   Female    21969
     Male      20206
31   Female    21588
     Male      20368
32   Female    20445
     Male      20254
33   Female    21603
     Male      20355
34   Female    23791
     Male      20177
35   Female    20594
     Male      20345
36   Female    20534
     Male      20200
37   Female    21130
     Male      24170
38   Female    21575
     Male      20968
39   Female    22021
     Male      21407
40   Female    20442
     Male      20733
41   Female    23563
 

Otro método groupby() digno de mención es agg(), que te permite ejecutar un montón de funciones diferentes en el DataFrame simultáneamente. Por ejemplo, podemos generar un simple resumen estadístico del conjunto de datos de la siguiente manera:


In [127]:
compras.groupby('Gender')['Annual Income'].agg(['count', 'min', 'max'])

,count,min,max
Gender,,,
Female,7484,20022,199963
Male,7595,20026,199974


### **4. Uso de multi-índices**


En todos los ejemplos que hemos visto hasta ahora hemos estado trabajando con objetos DataFrame o Series con un índice de una sola etiqueta. groupby() es ligeramente diferente en el hecho de que, dependiendo de la operación que ejecutemos, a veces dará como resultado lo que se denomina un multi-índice.

Un multi-índice difiere de un índice normal en que tiene varios niveles. Por ejemplo:


In [131]:
salario_por_edad_y_genero = compras.groupby(['Age', 'Gender'])['Annual Income'].mean().head(50)
salario_por_edad_y_genero

Age  Gender
18   Female    115839.533333
     Male      117176.182692
19   Female    115751.666667
     Male      101830.598039
20   Female    119786.554545
     Male      112534.117647
21   Female    107881.632075
     Male       97825.752381
22   Female    115143.904762
     Male      106193.491228
23   Female    113876.855556
     Male      110721.023529
24   Female    107264.806818
     Male      114993.617647
25   Female    108809.852632
     Male      109450.030612
26   Female    104941.183486
     Male      118056.109091
27   Female    114880.353535
     Male      116129.144231
28   Female    103319.397849
     Male      106394.510204
29   Female    107263.508929
     Male      114678.730769
30   Female    111414.555556
     Male      106172.050505
31   Female    112008.558824
     Male      108710.834783
32   Female    109157.154545
     Male      102505.132743
33   Female    117573.598039
     Male      110577.950495
34   Female    104076.295238
     Male      102780.342857
35

Los índices múltiples disponen de varios métodos para tratar su estructura escalonada que no existen en los índices de un solo nivel. También requieren dos niveles de etiquetas para recuperar un valor. Es un poco complicado al principio tratar con estas estructuras.

Los casos de uso de un multi-índice se detallan en la documentación de Pandas: https://pandas.pydata.org/pandas-docs/stable/user_guide/advanced.html.

Sin embargo, en general, el método multi-índice que se utiliza más a menudo es el que permite volver a un índice normal, el método reset_index():


In [132]:
salario_por_edad_y_genero.reset_index()

,Age,Gender,Annual Income
0,18,Female,115839.533333
1,18,Male,117176.182692
2,19,Female,115751.666667
3,19,Male,101830.598039
4,20,Female,119786.554545
5,20,Male,112534.117647
6,21,Female,107881.632075
7,21,Male,97825.752381
8,22,Female,115143.904762
9,22,Male,106193.491228


### **4. Uso de función sort**


Observando de nuevo el dataframe de `salario_por_edad_y_genero` podemos ver que la agrupación devuelve los datos en orden de índice, no en orden de valor. Es decir, al mostrar el resultado de un groupby, el orden de las filas depende de los valores del índice, no de los datos.

El método sort_values() es útil para esto. sort_values() utiliza por defecto una ordenación ascendente, en la que los valores más bajos van primero. Sin embargo, la mayoría de las veces queremos una ordenación descendente, donde los números más altos van primero, con el parámetro `ascending=False` lo conseguimos.

Por ejemplo, para obtener los datos de media de salario agrupados por edad y género podemos hacer lo siguiente.


In [135]:
media_salario_agg = compras.groupby(['Age', 'Gender'], as_index=False)['Annual Income'].mean()
media_salario_agg.sort_values(by='Annual Income').reset_index().drop('index', axis=1)

,Age,Gender,Annual Income
0,42,Female,93550.290000
1,73,Male,95970.686047
2,48,Male,97701.724138
3,21,Male,97825.752381
4,62,Female,97959.456311
...,...,...,...
141,55,Female,120120.240000
142,67,Female,120201.298246
143,81,Male,120580.787234
144,90,Male,121441.939655


Para ordenar por valores de índice, utilizamos el método complementario sort_index(). Este método tiene los mismos argumentos y orden por defecto:


In [136]:
media_salario_agg = compras.groupby(['Age', 'Gender'], as_index=False)['Annual Income'].mean()
media_salario_agg.sort_values(by='Annual Income').reset_index().sort_index()

,index,Age,Gender,Annual Income
0,48,42,Female,93550.290000
1,111,73,Male,95970.686047
2,61,48,Male,97701.724138
3,7,21,Male,97825.752381
4,88,62,Female,97959.456311
...,...,...,...,...
141,74,55,Female,120120.240000
142,98,67,Female,120201.298246
143,127,81,Male,120580.787234
144,145,90,Male,121441.939655


Por último, también podemos ordenar por más de una columna a la vez:


In [137]:
media_salario_agg.sort_values(by=['Age', 'Annual Income'], ascending=False)

,Age,Gender,Annual Income
145,90,Male,121441.939655
144,90,Female,109761.361905
142,89,Female,113670.909091
143,89,Male,113347.536364
140,88,Female,112899.379630
...,...,...,...
5,20,Male,112534.117647
2,19,Female,115751.666667
3,19,Male,101830.598039
1,18,Male,117176.182692
